# Supplemental figure: model performance across all three classifiers
A comprehensive companion to `12.assembled_figure.ipynb`'s headline figure (which focuses on confusion matrices and biological inference). This one assembles every training/held-out-performance panel saved by the three model pipelines in this module, nothing is recomputed here:

A. Binary control classifier: training curves (`3.plot_binary_mlp_control_performance.ipynb`)
B. Binary control classifier: precision-recall curves (`3.plot_binary_mlp_control_performance.ipynb`)
C. Treatment+dose multiclass classifier: training curve (`8.plot_multiclass_mlp_treatment_dose.ipynb`)
D. Treatment+dose multiclass classifier: top-k accuracy (`8.plot_multiclass_mlp_treatment_dose.ipynb`)
E. Treatment+dose multiclass classifier: per-class F1 (`8.plot_multiclass_mlp_treatment_dose.ipynb`)
F. Target/MOA multiclass classifier: training curve (`11.plot_multiclass_mlp_target.ipynb`)
G. Target/MOA multiclass classifier: top-k accuracy (`11.plot_multiclass_mlp_target.ipynb`)
H. Target/MOA multiclass classifier: per-class F1 (`11.plot_multiclass_mlp_target.ipynb`)

In [1]:
list_of_packages <- c("ggplot2", "patchwork", "magick", "grid", "RColorBrewer")
for (package in list_of_packages) {
    suppressPackageStartupMessages(
        suppressWarnings(
            library(package, character.only = TRUE, quietly = TRUE, warn.conflicts = FALSE)
        )
    )
}

find_git_root <- function() {
    cwd <- getwd()
    if (dir.exists(file.path(cwd, ".git"))) {
        return(cwd)
    }
    current_path <- cwd
    while (dirname(current_path) != current_path) {
        parent_path <- dirname(current_path)
        if (dir.exists(file.path(parent_path, ".git"))) {
            return(parent_path)
        }
        current_path <- parent_path
    }
    stop("No Git root directory found.")
}

root_dir <- find_git_root()
source(file.path(root_dir, "utils", "r_plot_themes.r"))
source(file.path(root_dir, "utils", "r_plot_funcs.r"))

module_dir <- file.path(root_dir, "6.treatment_predictive_model")
figures_dir <- file.path(module_dir, "figures")
dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)

## Embed pre-rendered PNGs
Each of these is already a finished, saved figure from an earlier notebook - no need to redraw them.

In [2]:
embed_png_panel <- function(path) {
    #' Reads a saved PNG and trims its (mostly white) border margin before
    #' wrapping it as a patchwork-compatible panel - each source figure was
    #' saved with its own surrounding whitespace, which otherwise stacks up
    #' when the panels are packed into a grid alongside each other.
    if (!file.exists(path)) {
        stop(
            "missing ", path, " - run the notebook that saves this figure first."
        )
    }
    img <- magick::image_trim(magick::image_read(path))
    wrap_elements(full = grid::rasterGrob(as.raster(img), interpolate = TRUE))
}

panel_a <- embed_png_panel(file.path(figures_dir, "binary_mlp_control_training_curves.png"))
panel_b <- embed_png_panel(file.path(figures_dir, "binary_mlp_control_pr_curves.png"))
panel_c <- embed_png_panel(file.path(figures_dir, "multiclass_mlp_treatment_dose_training_curve.png"))
panel_d <- embed_png_panel(file.path(figures_dir, "multiclass_mlp_treatment_dose_top_k_accuracy.png"))
panel_e <- embed_png_panel(file.path(figures_dir, "multiclass_mlp_treatment_dose_per_class_f1.png"))
panel_f <- embed_png_panel(file.path(figures_dir, "multiclass_mlp_target_training_curve.png"))
panel_g <- embed_png_panel(file.path(figures_dir, "multiclass_mlp_target_top_k_accuracy.png"))
panel_h <- embed_png_panel(file.path(figures_dir, "multiclass_mlp_target_per_class_f1.png"))

## Assemble
A 4x2 grid: row 1 is the binary control classifier (A, B) plus the treatment+dose classifier's training curve and top-k accuracy (C, D); row 2 is the treatment+dose classifier's per-class F1 (E) plus the target/MOA classifier's three panels (F, G, H).

In [3]:
fig_width <- 22
fig_height <- 11
layout <- "
ABCD
EFGH
"

dims <- set_plot_size(fig_width, fig_height)

supplemental_performance_figure <- (
    wrap_plots(
        A = panel_a, B = panel_b, C = panel_c, D = panel_d,
        E = panel_e, F = panel_f, G = panel_g, H = panel_h,
        design = layout
    )
    + plot_annotation(tag_levels = "A")
    & theme(plot.margin = margin(2, 2, 2, 2))
)

save_ggplot(
    supplemental_performance_figure,
    file.path(figures_dir, "supplemental_performance_figure.png"),
    width = dims$width, height = dims$height, dpi = 600
)